### Look at the bigger picture

In [4]:
import pandas as pd                
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, LogisticRegression 
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error, 
                           accuracy_score, f1_score, roc_auc_score, confusion_matrix,
                           classification_report)
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

# Look at the Bigger Picture
## 1. Titanic Survival Prediction
### 1.1 Frame the problem
* We have two options, yes or no. whether they survived or not. 
* Based on given parameters we have to decide how to measure success. i.e accuracy. 
* Manual assesment based on passenger class, age, gender
### 1.2 Select Performance Measure
* Accuracy, F1-Score, ROC-AUC 


## Get the Data

In [5]:
try:
    df = pd.read_csv('titanic/train.csv')
    print(f'data loaded! {df.shape[0]} rows and {df.shape[1]} columns')
except FileNotFoundError:
    print('titanic/train.csv not found. Please ensure the file exists.')
    



data loaded! 891 rows and 12 columns


## 3. Explore and Visualize data
### 3.1 Quick Overview
* Each row represent a different passenger with their attributes like:
 id, Pclass,Sex,Age,Sibsp,Parch,Ticket,Fare,Cabin and Embarked
 * info() gives quick description of data.
 * There are total 891 instances. Age, Cabin, Embarked have null values.
 
 

In [6]:
# Dataset Overview
print('Quick Overview')
print(f'Dataset Shape:  {df.shape}')
print(f'Datatypes: {df.dtypes}')

# Missing Values
print("=" * 20)
print('no of missing values: ')
missing = df.isnull().sum()
print(missing[missing>0])

#Stastical Summary
print("=" * 20)
print('Stastical Summary')
print(df.describe)
df.info()

Quick Overview
Dataset Shape:  (891, 12)
Datatypes: PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object
no of missing values: 
Age         177
Cabin       687
Embarked      2
dtype: int64
Stastical Summary
<bound method NDFrame.describe of      PassengerId  Survived  Pclass  \
0              1         0       3   
1              2         1       1   
2              3         1       3   
3              4         1       1   
4              5         0       3   
..           ...       ...     ...   
886          887         0       2   
887          888         1       1   
888          889         0       3   
889          890         1       1   
890          891         0       3   

                                                  Name     Sex   

## 3.2 Target Variable 

In [7]:
sur_count = df['Survived'].value_counts()
sur_rate = df['Survived'].mean()
sur_per = (sur_count[1]/len(df)*100)
nonSur_per = (sur_count[0]/len(df)*100 )
print(f'Survivors percentage: {sur_per:.1f}%')
print(f'Non-Survivors percentage: {nonSur_per:.1f}%')

Survivors percentage: 38.4%
Non-Survivors percentage: 61.6%


### 3.3 Feature Analysis
#### Survivors by:
* Passenger class
* Gender
* Age


In [8]:
pclass_sur = df.groupby('Pclass')['Survived'].agg(['count','sum','mean'])
print(f'Survivor by Pclass: {pclass_sur}')

print("="*20)
gender_sur = df.groupby('Sex')['Survived'].agg(['count','sum','mean'])
print(f'Survivor by gender: {gender_sur}')

print("="*20)
survivors = df[df['Survived']==1]['Age'].mean()
non_survivors = df[df['Survived']==0]['Age'].mean()
print(f'survivors: {survivors}')
print(f'non_survivors: {non_survivors}')

Survivor by Pclass:         count  sum      mean
Pclass                      
1         216  136  0.629630
2         184   87  0.472826
3         491  119  0.242363
Survivor by gender:         count  sum      mean
Sex                         
female    314  233  0.742038
male      577  109  0.188908
survivors: 28.343689655172415
non_survivors: 30.62617924528302


## 4. Prepare Data

In [9]:
age_missing = df['Age'].isnull().sum()
print(age_missing)

177


### 4.1 Handling Missing values 


In [10]:
#Handling missing AGE values
age_missing = df['Age'].isnull().sum()
cabin_missing = df['Cabin'].isnull().sum()
# print(f'Missing values of: \n Age: {age_missing} \n Cabin: {cabin_missing}')
group = df.groupby('Pclass')
for Pclass, Class_df in group:
    # print(Class_df[['Pclass','Age']].head(10))
    med=Class_df['Age'].median()
    # print(med)
    mask = (df['Pclass']==Pclass) & (df['Age'].isnull())
    df.loc[mask,'Age'] = med
    # print(df.loc[df['Pclass'] == Pclass, 'Age'].head(10))

#   df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])


df['Fare'] = df['Fare'].fillna(df['Fare'].median())


#encoding categorical features
# df = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True)

df = pd.get_dummies(df, columns=['Sex'], drop_first=True)
df.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Age', 'SibSp', 'Parch',
       'Ticket', 'Fare', 'Cabin', 'Embarked', 'Sex_male'],
      dtype='object')

### 4.2 Feature Engineering
* New feature:  family size, Age groups, Fare groups
* Extract title from Name
* Categorial variables encoding

In [11]:
df['Family_size'] = df['SibSp'] + df['Parch'] + 1
df['Age_group'] = pd.cut(df['Age'], 
                                 bins=[0, 12, 18, 35, 60, 100], 
                                 labels=['Child', 'Teen', 'Adult', 'Middle', 'Senior'])

df['Fare_group'] = pd.qcut(df['Fare'], q=4, labels=['Low', 'Medium', 'High', 'VeryHigh'])


# Extract title from name
df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
df['Title'] = df['Title'].replace(['Lady', 'Countess','Capt', 'Col',
                                                     'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Other')
df['Title'] = df['Title'].replace('Mlle', 'Miss')
df['Title'] = df['Title'].replace('Ms', 'Miss')
df['Title'] = df['Title'].replace('Mme', 'Mrs')

#Categorical encoding
categorical_features = [ 'Embarked', 'Age_group', 'Fare_group', 'Title']
df_encoded = pd.get_dummies(df, columns=categorical_features, prefix=categorical_features, drop_first=True)
print(f"New shape: {df_encoded.shape}")


New shape: (891, 25)


### 5. Select and Train Model

#### defining dependent and independent variables for linear regression

In [12]:
# linear regression predict fare
X_lin = df[['Pclass', 'SibSp', 'Parch', 'Sex_male']]
y_lin = df['Age']


#### Defining dependent and independent variables for linear regression

In [13]:
# logistic regression predict survived
X_log = df[['Pclass', 'Age', 'SibSp', 'Parch', 'Sex_male']]
y_log = df['Survived']


#### Splitting data set into train and test set for linear and logistic regression.

In [14]:
# split dataset
X_train_lin, X_test_lin, y_train_lin, y_test_lin = train_test_split(X_lin, y_lin, test_size=0.24,random_state=42)
X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(X_log, y_log, test_size=0.24, random_state=42)
linear_model = LinearRegression()
logistic_model = LogisticRegression()

# column_names = ['Pclass', 'Age', 'SibSp', 'Parch', 'Sex_male']

# X_train_log = pd.DataFrame(X_train_log, columns=column_names)
# X_test_log = pd.DataFrame(X_test_log, columns=column_names)

#### Standardize data using StandardScaler()

In [15]:
# Standardization of features
scaler_lin=StandardScaler()
X_train_lin_scl = scaler_lin.fit_transform(X_train_lin)
X_test_lin_scl = scaler_lin.transform(X_test_lin)

scaler_log = StandardScaler()
X_train_log_scl = scaler_log.fit_transform(X_train_log)
X_test_log_scl = scaler_log.transform(X_test_log)


In [19]:
# predict 
linear_model.fit(X_train_lin, y_train_lin)
y_pred_lin = linear_model.predict(X_test_lin)

logistic_model.fit(X_train_log, y_train_log)
y_pred_log = logistic_model.predict(X_test_log)


In [20]:
print("LinearRegression : ")
print("R square", r2_score(y_test_lin, y_pred_lin))
print("RMSE:", np.sqrt(mean_squared_error(y_test_lin, y_pred_lin)))
print("MAE:", mean_absolute_error(y_test_lin, y_pred_lin))


LinearRegression : 
R square 0.21848746439335753
RMSE: 11.720518166026318
MAE: 8.464280424712262


In [21]:
y_test_log = y_test_log[:len(X_test_log)]

print("\nLogistic Regression :")
print("Accuracy:", accuracy_score(y_test_log, y_pred_log))
print("F1 Score:", f1_score(y_test_log, y_pred_log))
print("ROC-AUC:", roc_auc_score(y_test_log, logistic_model.predict_proba(X_test_log)[:,1]))
print("Confusion Matrix:\n", confusion_matrix(y_test_log, y_pred_log))




Logistic Regression :
Accuracy: 0.8177570093457944
F1 Score: 0.7636363636363637
ROC-AUC: 0.8793103448275862
Confusion Matrix:
 [[112  15]
 [ 24  63]]


## Finetune model

In [22]:
# Cross validation for model evaluation
print('Cross validation: ')
lin_cv_scores = cross_val_score(linear_model, X_train_lin, y_train_lin, cv=5, scoring='r2')
print(f"Linear Regression R² scores: {lin_cv_scores}")
print(f"Linear Regression R² mean: {lin_cv_scores.mean():.4f} (+/- {lin_cv_scores.std() * 2:.4f})")

log_cv_scores = cross_val_score(logistic_model, X_train_log, y_train_log, cv=5, scoring='accuracy')
print(f"Logistic Regression Accuracy scores: {log_cv_scores}")
print(f"Logistic Regression Accuracy mean: {log_cv_scores.mean():.4f} (+/- {log_cv_scores.std() * 2:.4f})")


Cross validation: 
Linear Regression R² scores: [0.27194555 0.09859869 0.355281   0.19609017 0.08031685]
Linear Regression R² mean: 0.2004 (+/- 0.2077)
Logistic Regression Accuracy scores: [0.79411765 0.85294118 0.78518519 0.73333333 0.83703704]
Logistic Regression Accuracy mean: 0.8005 (+/- 0.0842)


In [24]:
# Hyper parameter tuning for logistic regression
print('LogisticRegression Hyperparameter Tuning')
log_para_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'lbfgs'],
    'max_iter': [1000, 2000]
}

log_grid_search = GridSearchCV(
    LogisticRegression(random_state=42),
    log_para_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)

log_grid_search.fit(X_train_log, y_train_log)
print(f"Selected parameters for Logistic Regression: {log_grid_search.best_params_}")
print(f"Selected CV score for Logistic Regression: {log_grid_search.best_score_:.4f}")

updated_logistic_model = log_grid_search.best_estimator_
y_pred_log_tuned = updated_logistic_model.predict(X_test_log)

LogisticRegression Hyperparameter Tuning
Selected parameters for Logistic Regression: {'C': 1, 'max_iter': 1000, 'solver': 'lbfgs'}
Selected CV score for Logistic Regression: 0.8005


## Present your solution


In [25]:
# 7.1 Linear Regression Results
print(" Linear Regression Results:")
print("LinearRegression:")
print("R square:", r2_score(y_test_lin, y_pred_lin))
print("RMSE:", np.sqrt(mean_squared_error(y_test_lin, y_pred_lin)))
print("MAE:", mean_absolute_error(y_test_lin, y_pred_lin))

# 7.2 Logistic Regression Results (Original)
print("7.2 Logistic Regression  (Original):")
y_test_log_adjusted = y_test_log[:len(X_test_log)]

print("Accuracy:", accuracy_score(y_test_log_adjusted, y_pred_log))
print("F1 Score:", f1_score(y_test_log_adjusted, y_pred_log))
print("ROC-AUC:", roc_auc_score(y_test_log_adjusted, logistic_model.predict_proba(X_test_log)[:,1]))
print("Confusion Matrix:\n", confusion_matrix(y_test_log_adjusted, y_pred_log))

# 7.3 Logistic Regression Results (Tuned)
print("7.3 Logistic Regression Results (Tuned):")
print("Accuracy:", accuracy_score(y_test_log_adjusted, y_pred_log_tuned))
print("F1 Score:", f1_score(y_test_log_adjusted, y_pred_log_tuned))
print("ROC-AUC:", roc_auc_score(y_test_log_adjusted, updated_logistic_model.predict_proba(X_test_log)[:,1]))
print("Confusion Matrix:\n", confusion_matrix(y_test_log_adjusted, y_pred_log_tuned))


 Linear Regression Results:
LinearRegression:
R square: 0.21848746439335753
RMSE: 11.720518166026318
MAE: 8.464280424712262
7.2 Logistic Regression  (Original):
Accuracy: 0.8177570093457944
F1 Score: 0.7636363636363637
ROC-AUC: 0.8793103448275862
Confusion Matrix:
 [[112  15]
 [ 24  63]]
7.3 Logistic Regression Results (Tuned):
Accuracy: 0.8177570093457944
F1 Score: 0.7636363636363637
ROC-AUC: 0.8793103448275862
Confusion Matrix:
 [[112  15]
 [ 24  63]]


### The Original and fine tuned are identical. so default parameters are likely to be optimal.